# 15 · DeBERTa-v3-base ce4, seed 43, with a saved checkpoint (supplement T3)

**Why:** (1) none of the DeBERTa runs saved its weights, so the negation probe (notebook 16) and the easy-query holdout (notebook 17) need a DeBERTa checkpoint; (2) a second seed of the base ce4 configuration, so that ce4 vs MSE on DeBERTa has two seeds on each side and the seed noise of the final system's backbone is measured.

**Configuration** = `d2_deberta_main_ce4_s42` (notebooks 06 / 07): `microsoft/deberta-v3-base`, `main`, `max_len` 256, ce4, lr 2e-5, 1,200 warm-up steps, 1 epoch, batch 32, fp16; **seed 43**; random pair batches as in the original run (`sampler` not set).

**Checks:** val nDCG within 0.8891 ± 0.003 (seed 42); paired differences with `d2_deberta_main_ce4_s42` and `d1_deberta_main_mse_s43` if their val scores are in the inputs. The checkpoint is saved in fp16 after predicting and reloaded once to verify that it reproduces the val scores.

**Kaggle settings:** GPU **T4** (one GPU), **Internet on**. About 2.8 h.

**Inputs:**
- Dataset `wm-src` built from the **`supplement`** branch. Required. Its `data/results/scores/` may hold the val scores of `d2_deberta_main_ce4_s42` and `d1_deberta_main_mse_s43` (paired comparisons; skipped otherwise).
- Add Input → Notebook Output → **01_preprocess**. Required.

**How to run.** `MODE = 'smoke'` (5 % of train, val only, checkpoint save and reload checked, about 15 minutes), then `MODE = 'full'` → Save & Run All. Download `results/` into `data/results/supplement/`; the notebook output (`models/d2_deberta_main_ce4_s43/`) is the checkpoint input of notebooks 16 and 17.

**Output** (`/kaggle/working/`): `results/d2_deberta_main_ce4_s43.json`, `results/scores/d2_deberta_main_ce4_s43_{val,test}.parquet` (with `p_E..p_I`), `models/d2_deberta_main_ce4_s43/`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, json, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.cross_encoder import DEVICE, predict, train_ce
from src.data import load_pairs
from src.evaluation import RunLog, calibration, paired_bootstrap

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train, val only | 'full'

CFG = {'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256, 'loss': 'ce4',
       'lr': 2e-5, 'warmup_steps': 1200, 'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01,
       'max_grad_norm': 1.0, 'fp16': True, 'eval_batch_size': 64}      # = d2_deberta_main_ce4_s42
SEED = 43
REFS = ['d2_deberta_main_ce4_s42', 'd1_deberta_main_mse_s43']
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']     # test is predicted and saved, never scored here
EVAL_EVERY = 300 if MODE == 'smoke' else None
NAME = f"{'smoke_' if MODE == 'smoke' else ''}d2_deberta_main_ce4_s{SEED}"
RES, MODELS = Path('/kaggle/working/results'), Path('/kaggle/working/models')

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| run', NAME)

In [ ]:
pairs = load_pairs(CFG['fields'])
train_df = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=42)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[pairs['split'] == s].reset_index(drop=True) for s in EVAL_SPLITS}
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs')

In [ ]:
log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
refs = [r for r in REFS if MODE == 'full' and log.load(r, splits=['val'])]

model, tokenizer, info = train_ce(CFG, train_df, SEED, eval_df=parts['val'], eval_every=EVAL_EVERY,
                                  print_every=100 if MODE == 'smoke' else 1000)
t0 = time.time()
torch.cuda.empty_cache()
out = {s: predict(model, tokenizer, parts[s], CFG['max_len'], batch_size=CFG['eval_batch_size'], fp16=CFG['fp16'],
                  return_probs=True) for s in EVAL_SPLITS}
rec = log.log(NAME, {s: o[0] for s, o in out.items()},
              {'mode': MODE, **CFG, 'seed': SEED, 'subsample': SUBSAMPLE,
               'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
               'predict_pairs': int(sum(len(parts[s]) for s in EVAL_SPLITS)), 'gpu': torch.cuda.get_device_name(0),
               'n_gpus': 1, 'predict_minutes': round((time.time() - t0) / 60, 2), **info},
              compare_to=refs[0] if refs else None, blind=('test',), extra={s: o[1] for s, o in out.items()})
for r in refs[1:]:
    print(f'val {NAME} - {r}:', paired_bootstrap(log.per_query[r]['val'], log.per_query[NAME]['val']))
print('calibration (val):', calibration(out['val'][1].to_numpy(), parts['val']['esci_label'].to_numpy()))

## Save the checkpoint and check it

Saved in fp16 after predicting (`half()` changes the model in place). The checkpoint is reloaded and re-scores 2,000 val pairs; the scores must match the logged ones (fp16 tolerance).

In [ ]:
d = MODELS / NAME
model.half().save_pretrained(d)
tokenizer.save_pretrained(d)
(d / 'checkpoint_info.json').write_text(json.dumps(
    {'run': NAME, 'cfg': CFG, 'seed': SEED, 'val_ndcg': rec['metrics']['val']['ndcg'], 'src_version': src_version()},
    indent=1))
del model
gc.collect()
torch.cuda.empty_cache()

m2 = AutoModelForSequenceClassification.from_pretrained(d, torch_dtype=torch.float16).to(DEVICE)
t2 = AutoTokenizer.from_pretrained(d)
sub = parts['val'].iloc[:2000]
again = predict(m2, t2, sub, CFG['max_len'], batch_size=CFG['eval_batch_size'], fp16=True)
gap = float(np.abs(again - out['val'][0][:2000]).max())
size_mb = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 2**20
print(f'checkpoint {d} | {size_mb:.0f} MB | max |score difference| after reload {gap:.4f}')
assert gap < 0.02, 'the reloaded checkpoint does not reproduce the scores'
del m2

In [ ]:
v = rec['metrics']['val']['ndcg']
if MODE == 'smoke':
    first, last = rec['loss_log'][0]['loss'], rec['loss_log'][-1]['loss']
    ok = last < first and v > 0.742
    print(f"{'OK   ' if ok else 'CHECK'} loss {first:.4f} -> {last:.4f} | val nDCG {v} | "
          f"full epoch ~{rec['steps'] / SUBSAMPLE / rec['it_per_s'] / 3600:.2f} h")
    print('Smoke test passed: switch MODE to full.' if ok else 'Training did not learn: send me this output.')
else:
    print(f"{'OK   ' if abs(v - 0.8891) <= 0.003 else 'CHECK'} val nDCG {v} (seed 42: 0.8891; expected within 0.003)")